# XGBoost 再学習ノートブック v6 — リーク修正 + base_margin パリティ対応

**このノートは上から順に全セルを実行するだけで完了します。**（所要 30〜60分）

## v5 からの変更点

| 内容 | 効果 |
|---|---|
| 学習データのリーク3系統を除去 | `f_early_speed`(実測ペース) / `running_style`(実測脚質) / `member_level`(未来参照) |
| `f_pred_gap_*` 5列を削除 | 「乖離は繰り返す」前提が実データで否定された（相関 -0.066） |
| `corner_all` を学習側にも供給 | `f_corner_position_change` が定数だったのを修正 |
| **base_margin に実測ドリフトを注入** | 学習=確定人気 / 推論=朝の薄い人気 というパリティ違反を是正 |

## 実行前の確認

- Colab の「シークレット」に `GITHUB_PAT` が登録されていること（最終セルのpushで使用）
- ランタイムは CPU で可

⚠ セル4の自動検証が **FAIL** を出したら、そこで止めて内容を確認してください。
その先に進むと壊れたモデルを本番へ反映してしまいます。


In [ ]:
# == セル1: セットアップ ===================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'BASE_DIR: {BASE_DIR}')


In [ ]:
# == セル2: src/ 強制アップデート（GitHub最新コードを取得）==================
# ⚠ v5から shap_explain.py / rank_matrix_filter.py を追加（v5では欠落しており
#    Colab側が古いままになっていた）
import urllib.request, time as _time

# 取得元ブランチ。通常は 'main'。
# 未マージの作業ブランチから取る場合のみ 'refs/heads/<branch>' を指定する
# （スラッシュを含むブランチ名でも refs/heads/ 形式なら正しく解決される）。
BRANCH = 'main'
BASE_URL = f'https://raw.githubusercontent.com/hanagenuku/keiba_ai/{BRANCH}'
print(f'取得元: {BRANCH}')
_cb = int(_time.time())

files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py', 'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py', 'src/tools/rescrape_history.py',
    'src/tools/build_training_data.py', 'src/tools/train_xgb.py',
    'src/tools/calibrate_xgb.py', 'src/tools/generate_style_advantage.py',
    'src/tools/train_pace_model.py', 'src/tools/shap_diagnosis.py',
    'src/features/engine.py', 'src/features/speed_index.py',
    'src/features/horse_type.py', 'src/features/error_tags.py',
    'src/features/shap_explain.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py',
    'src/models/calibration_xgb.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py', 'src/betting/ev_filter.py',
    'src/betting/app_json.py', 'src/betting/race_simulator.py',
    'src/betting/ev_calculator.py',
    'src/betting/payout_estimator.py',
    'src/betting/bet_optimizer.py', 'src/betting/shadow.py',
    'src/betting/rank_matrix_filter.py',
]
data_files = [
    'data/course_profiles.json', 'data/course_distance_profiles.json',
    'data/note_schema.json',
]

_failed = []
for rel in files + data_files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    for _retry in range(3):
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
            print(f'OK   {rel}')
            break
        except Exception as _e:
            if _retry < 2:
                _time.sleep(2 ** _retry)
            else:
                print(f'FAIL {rel}: {_e}')
                _failed.append(rel)

# モジュールキャッシュを破棄して最新を読み込ませる
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

assert not _failed, f'取得に失敗したファイルがあります: {_failed}'

# 今回の修正が入ったコードか確認（古いキャッシュのまま進むのを防ぐ）
import inspect
from src.tools.train_xgb import train_xgb, load_popularity_drift
from src.tools.build_training_data import build_training_data
assert 'simulate_serving_popularity' in inspect.signature(train_xgb).parameters, (
    'train_xgb が古いままです。BRANCH の指定を確認してください'
    f'（現在: {BRANCH}）。main にまだマージされていない可能性があります。')
import src.tools.build_training_data as _btd
assert 'corner_all' in inspect.getsource(_btd), 'build_training_data が古い'
print('\n✅ 最新コードの取得を確認しました')


In [ ]:
# == セル3: GitHub→Drive データマージ（history.db の最新化）================
# 週次ワークフローがGitHub側の history.db を更新しているので取り込む。
import sqlite3, urllib.request, os

os.makedirs(f'{BASE_DIR}/data', exist_ok=True)
gh_db = f'{BASE_DIR}/data/_history_github.db'
url = 'https://media.githubusercontent.com/media/hanagenuku/keiba_ai/main/data/history.db'
urllib.request.urlretrieve(url, gh_db)
print(f'GitHub history.db 取得: {os.path.getsize(gh_db):,} bytes')

local_db = f'{BASE_DIR}/data/history.db'

# Drive側のスキーマをGitHub側に揃える（空リストでALTER TABLEだけ適用）
from src.utils.db import save_history_db
save_history_db([], base_dir=BASE_DIR)

def merge(table):
    c = sqlite3.connect(local_db)
    c.execute(f"ATTACH DATABASE '{gh_db}' AS gh")
    # ⚠ SQLiteのPRAGMAはスキーマ名をpragma名の前に置く。
    #    PRAGMA table_info(gh.x) は構文エラーになる。
    info_loc = list(c.execute(f'PRAGMA main.table_info({table})'))
    loc = {r[1] for r in info_loc}
    rem = {r[1] for r in c.execute(f'PRAGMA gh.table_info({table})')}

    # 🔴 自動採番の主キー(INTEGER PRIMARY KEY = rowid別名)は絶対に持ち込まない。
    #    GitHub側とDrive側は別系統でidを採番しているため、idごとINSERTすると
    #    主キー衝突→ INSERT OR IGNORE が黙って行を捨てる。
    #    実際これで horse_history が 74,680件中 7,819件しか入らず、
    #    「race_history にレースはあるのに馬が0頭」という状態を作っていた。
    #    重複排除は UNIQUE(race_id, horse_num) が担うのでidは不要。
    autopk = {r[1] for r in info_loc if r[5] and (r[2] or '').upper() == 'INTEGER'}
    common = sorted((loc & rem) - autopk)
    only_gh = sorted(rem - loc)
    if only_gh:
        print(f'  ⚠ {table}: GitHub側にしか無い列（今回は取り込まず）: {only_gh}')
    if autopk:
        print(f'  {table}: 自動採番PK {sorted(autopk)} はマージ対象から除外')
    before = c.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    cols = ','.join(f'"{x}"' for x in common)
    c.execute(f'INSERT OR IGNORE INTO {table} ({cols}) SELECT {cols} FROM gh.{table}')
    c.commit()
    after = c.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table}: {before:,} → {after:,} (+{after-before:,})')
    c.close()

for t in ('race_history', 'horse_history'):
    merge(t)

# ── 整合性チェック: 「レースはあるのに馬がいない」を検出する ──────────────
# ここが0でないと学習データが欠け、統合テストも 0頭 で落ちる。
c = sqlite3.connect(local_db)
orphan = c.execute("""
    SELECT COUNT(*) FROM race_history r
    WHERE NOT EXISTS (SELECT 1 FROM horse_history h WHERE h.race_id = r.race_id)
""").fetchone()[0]
total = c.execute('SELECT COUNT(*) FROM race_history').fetchone()[0]
recent = c.execute("""
    SELECT r.race_id, r.date FROM race_history r
    WHERE NOT EXISTS (SELECT 1 FROM horse_history h WHERE h.race_id = r.race_id)
    ORDER BY r.date DESC LIMIT 5
""").fetchall()
c.close()
print(f'\n整合性: 馬が1頭もいないレース {orphan} / {total:,} 件')
if recent:
    print('  例:', ', '.join(f'{a}({b})' for a, b in recent))
assert orphan == 0, (
    'race_history にあるのに horse_history に馬がいないレースがあります。'
    'マージが正しく効いていません（上の除外列を確認してください）。')
print('✅ race_history と horse_history が一致しています')

# keiba.db も取得（base_margin ドリフト分布の作成に odds_snapshots が要る）
try:
    urllib.request.urlretrieve(
        'https://media.githubusercontent.com/media/hanagenuku/keiba_ai/main/data/keiba.db',
        f'{BASE_DIR}/data/keiba.db')
    k = sqlite3.connect(f'{BASE_DIR}/data/keiba.db')
    n = k.execute('SELECT COUNT(*) FROM odds_snapshots').fetchone()[0]
    k.close()
    print(f'\nkeiba.db 取得: odds_snapshots {n:,} 行')
except Exception as e:
    print(f'\n⚠ keiba.db 取得失敗（ドリフト注入はスキップされます）: {e}')


In [ ]:
# == セル4: 学習データ再生成 + 修正が効いているかの自動検証 ================
import importlib, sys
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.features.speed_index import rebuild_speed_index_cache
try:
    rebuild_speed_index_cache(BASE_DIR)
    print('speed_index キャッシュ再構築 完了\n')
except Exception as e:
    print(f'speed_index 再構築スキップ: {e}\n')

from src.tools.build_training_data import build_training_data
build_training_data(BASE_DIR)

# ── 自動検証: 今回の修正がすべて反映されているか ──
import pandas as pd
df = pd.read_csv(f'{BASE_DIR}/data/horse_features.csv')
cols = list(df.columns)
checks = []

checks.append(('member_level 系が消えている',
               len([c for c in cols if 'member_level' in c]) == 0))
checks.append(('pred_gap 系が消えている',
               len([c for c in cols if 'pred_gap' in c]) == 0))
checks.append(('f_early_speed がリークしていない（定数36.0）',
               set(df['f_early_speed'].dropna().unique()) == {36.0}))
checks.append(('f_corner_position_change が定数でない（corner_all供給OK）',
               df['f_corner_position_change'].nunique() > 1))

print(f'\n{"="*60}\n学習データ検証: {len(df):,}行 × {len(cols)}列\n{"="*60}')
ok = True
for name, passed in checks:
    print(f'  {"PASS" if passed else "FAIL"}  {name}')
    ok &= passed
assert ok, '検証に失敗しました。ここで止めて内容を確認してください。'
print('\n✅ すべての修正が反映されています')


In [ ]:
# == セル5: 残差モデル再学習（base_margin ドリフト注入ON）==================
# simulate_serving_popularity=True（既定）で、確定人気に実測ドリフトを注入し
# 「推論時に届く朝の人気」と同じ情報量で学習する。
import sys
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.tools.train_xgb import train_xgb, load_popularity_drift

drift = load_popularity_drift(BASE_DIR)
if drift is None:
    print('⚠ ドリフト標本が不足しています（odds_snapshots が薄い）。')
    print('  従来どおり確定人気で学習します。数週間データが溜まってから再実行を推奨。')
else:
    print(f'ドリフト分布: {len(drift)-1}人気分 / 標本 {len(drift["_all"]):,} 件\n')

res_new = train_xgb(BASE_DIR, residual=True, simulate_serving_popularity=True)
print(f'\n残差モデル(ドリフト注入ON) Val AUC = {res_new["auc"]:.4f}')


In [ ]:
# == セル6: 比較（ドリフト注入 ON vs OFF）==================================
# 効果を数値で確認する。OFF側は比較専用で本番には反映しない。
# ⚠ フル学習をもう一度回すのでセル5と同じだけ時間がかかる（正常）。
import shutil, os, json as _json
from datetime import datetime, timezone, timedelta

# ON側の成果物を退避（OFF学習で上書きされるため）
for f in ['xgb_fukusho_model_residual.pkl', 'xgb_feature_cols_residual.json']:
    p = f'{BASE_DIR}/data/{f}'
    if os.path.exists(p):
        shutil.copy2(p, p + '.drift_on')

res_off = train_xgb(BASE_DIR, residual=True, simulate_serving_popularity=False)

# ON側を復元
for f in ['xgb_fukusho_model_residual.pkl', 'xgb_feature_cols_residual.json']:
    p = f'{BASE_DIR}/data/{f}'
    if os.path.exists(p + '.drift_on'):
        shutil.move(p + '.drift_on', p)

print(f'\n{"="*60}')
print(f'  ドリフト注入 OFF (従来): Val AUC = {res_off["auc"]:.4f}')
print(f'  ドリフト注入 ON  (今回): Val AUC = {res_new["auc"]:.4f}')
print(f'{"="*60}')
print('\n※ この Val AUC は「確定人気で評価」した値のため、ONの方が低く出るのが正常です。')
print('  ONの真価は、本番と同じ「朝の人気」で推論したときに現れます')
print('  （オフライン検証では 複勝AUC +0.004 / 単勝AUC +0.006 / 較正後logloss -0.015）。')

# ── ★ 公平な比較: 本番と同じ「朝の人気」条件で ON/OFF を評価 ──────────
# ⚠ 上の Val AUC は確定人気で評価しているため **OFF が有利に出る**。
#   その数字だけで OFF を選ぶと、本番では手に入らない情報で
#   モデルを選んだことになる。判断はこちらの数字で行うこと。
#   （2026-07-31 実測: 確定人気では OFF が +0.0255 で勝つが、
#     朝の人気では ON が +0.0140 で勝ち、**符号が反転する**）
import numpy as _np, pandas as _pd, xgboost as _xgb
from sklearn.metrics import roc_auc_score as _auc
from src.tools.train_xgb import (load_popularity_drift as _ld,
                                 _apply_popularity_drift as _apd,
                                 _popularity_to_base_margin as _p2bm)

fair = {}
try:
    _df = _pd.read_csv(f'{BASE_DIR}/data/horse_features.csv')
    _df['_d'] = _pd.to_datetime(_df['date'].astype(str).str.replace('-', '', regex=False).str[:8],
                                format='%Y%m%d', errors='coerce')
    _v = _df.dropna(subset=['_d'])
    _v = _v[(_v['_d'] >= '2026-04-01') & (_v['_d'] <= '2026-05-31')].copy()
    _v['_n'] = _v.groupby('race_id')['horse_num'].transform('count')
    _pop = _v['f_popularity'].fillna(_v['_n'] / 2)      # train_xgb と同じ補完
    _cols = _json.load(open(f'{BASE_DIR}/data/xgb_feature_cols_residual.json'))['feature_cols']
    _X, _y = _v[_cols].fillna(5.0), _v['is_fukusho']
    _drift = _ld(BASE_DIR)
    if _drift is None:
        raise RuntimeError('ドリフト標本が不足')

    _M = {'on': 'xgb_fukusho_model_residual.pkl',       # セル6冒頭で復元済み
          'off': 'xgb_fukusho_model_residual_new.pkl'}  # OFF学習の生成物
    _b = {}
    for _k, _f in _M.items():
        _bb = _xgb.Booster(); _bb.load_model(f'{BASE_DIR}/data/{_f}'); _b[_k] = _bb

    _acc = {'on': [], 'off': []}
    for _s in (12, 101, 202, 303, 404):
        _bm = _p2bm(_pd.Series(_apd(_pop, _v['race_id'].values, _drift, seed=_s)), _v['_n'])
        _d = _xgb.DMatrix(_X, feature_names=list(_cols)); _d.set_base_margin(_bm)
        for _k in _M:                                    # 両モデルに同一の base_margin
            _p = 1 / (1 + _np.exp(-_b[_k].predict(_d, output_margin=True)))
            _acc[_k].append(_auc(_y, _p))
    fair = {'auc_on': round(float(_np.mean(_acc['on'])), 4),
            'auc_off': round(float(_np.mean(_acc['off'])), 4),
            'seeds': [12, 101, 202, 303, 404],
            'n_val_rows': int(len(_v)), 'n_val_races': int(_v['race_id'].nunique())}
    fair['delta_on_minus_off'] = round(fair['auc_on'] - fair['auc_off'], 4)

    print(f'\n{"="*60}')
    print('  ★ 本番と同じ条件（朝の人気）での比較 ← 判断はこちら')
    print(f'    ドリフト注入 ON  = {fair["auc_on"]:.4f}')
    print(f'    ドリフト注入 OFF = {fair["auc_off"]:.4f}')
    print(f'    差 (ON - OFF)    = {fair["delta_on_minus_off"]:+.4f}')
    print(f'{"="*60}')
    if fair['delta_on_minus_off'] <= -0.002:
        print('  ⚠ OFF の方が良い。xgb_fukusho_model_residual_new.pkl を')
        print('     本番にコピーして押し直すこと（学習は不要／較正は要再作成）。')
    else:
        print('  ✅ ON を採用してよい（このままセル7へ）。')
except Exception as _e:
    print(f'\n⚠ 公平比較をスキップ: {type(_e).__name__}: {_e}')

# ── 結果をファイルに残す ────────────────────────────────────────────────
# 出力セルは実行のたびに流れて読めなくなるので、必ずファイルに落とす。
# セル9でGitHubにpushするため、あとから第三者が確認できる。
JST = timezone(timedelta(hours=9))
report = {
    'trained_at': datetime.now(JST).strftime('%Y-%m-%d %H:%M:%S JST'),
    'notebook': 'KEIBA_XGB_retrain_v6',
    'drift_on':  {k: res_new.get(k) for k in
                  ('auc', 'brier', 'logloss', 'n_features', 'n_train', 'n_val')},
    'drift_off': {k: res_off.get(k) for k in
                  ('auc', 'brier', 'logloss', 'n_features', 'n_train', 'n_val')},
    'auc_delta_on_minus_off_confirmed_pop': round(res_new['auc'] - res_off['auc'], 4),
    'fair_comparison_serving_pop': fair,   # ★ 判断はこちらの数字で行う
    'note': ('drift_on/drift_off の Val は確定人気での評価なので OFF が有利に出る。'
             'モデル選択は fair_comparison_serving_pop（朝の人気で評価）で行うこと。'),
}
with open(f'{BASE_DIR}/data/retrain_v6_report.json', 'w') as f:
    _json.dump(report, f, ensure_ascii=False, indent=2)
print(f'\n📄 結果を保存: data/retrain_v6_report.json')
print(_json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
# == セル7: 本番切替 =======================================================
# 残差モデルを本番ファイル名にコピーし、キャリブレータを再作成する。
import shutil, json, os, sys

# 旧モデルをバックアップ
for f in ['xgb_fukusho_model.pkl', 'xgb_feature_cols.json', 'xgb_calibrator.pkl']:
    src = f'{BASE_DIR}/data/{f}'
    if os.path.exists(src):
        shutil.copy2(src, f'{src}.bak_v6')
        print(f'  バックアップ: {f} → {f}.bak_v6')

shutil.copy2(f'{BASE_DIR}/data/xgb_fukusho_model_residual.pkl',
             f'{BASE_DIR}/data/xgb_fukusho_model.pkl')
shutil.copy2(f'{BASE_DIR}/data/xgb_feature_cols_residual.json',
             f'{BASE_DIR}/data/xgb_feature_cols.json')

with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    meta = json.load(f)
assert meta.get('residual') is True, 'residual フラグがありません'
n_feat = len(meta['feature_cols'])
assert not [c for c in meta['feature_cols'] if 'member_level' in c or 'pred_gap' in c], \
    '削除したはずの列がモデルに残っています'
print(f'\n  residual={meta["residual"]}  特徴量数={n_feat}  Val AUC={meta.get("val_auc", 0):.4f}')

# ── キャリブレータ再作成（特徴量が変わったので必須）──────────────────────
# ⚠ 旧版はここが失敗しても「完了」と表示していた。
#    run_xgb_calibration は例外を投げず None を返す作りなので、
#    戻り値を必ず確認する。古い較正器が新モデルに付いたまま残ると、
#    cal_prob（表示用の複勝確率）だけが静かに誤った値になる。
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

cal = None
try:
    from src.tools.calibrate_xgb import run_xgb_calibration
    cal = run_xgb_calibration(BASE_DIR)
except Exception as e:
    print(f'\n  ⚠ キャリブレーション実行中に例外: {e}')

if cal is not None:
    print('\n  ✅ キャリブレーション再作成 完了')
else:
    # 新モデルに古い較正器を付けたままにしない。較正器が無ければ engine は
    # 生シグモイド確率を使う（順位・買い目には影響しない）。
    p = f'{BASE_DIR}/data/xgb_calibrator.pkl'
    if os.path.exists(p):
        shutil.move(p, p + '.stale_v6')
        print(f'\n  ⚠ キャリブレーション失敗。古い較正器は新モデルと不整合なので退避しました')
        print(f'     {os.path.basename(p)} → {os.path.basename(p)}.stale_v6')
    print('     cal_prob は生シグモイド確率になります（RL順位・買い目には影響なし）。')
    print('     ⚠ 原因を確認してください。ここが直らないと複勝確率の表示が甘くなります。')


In [ ]:
# == セル8: 統合テスト =====================================================
# 本番と同じ経路で1レース分の推論を通し、壊れていないことを確認する。
import sys, sqlite3
for key in list(sys.modules.keys()):
    if key.startswith('src'):
        del sys.modules[key]

from src.features.engine import init_engine, calc_all
import src.features.engine as eng

init_engine(BASE_DIR)
print(f'  _XGB_RESIDUAL = {eng._XGB_RESIDUAL}  (True であること)')
assert eng._XGB_RESIDUAL is True, '残差モデルとして認識されていません'

# history.db から実在の1レースを取り出して推論。
# ⚠ 「race_history の最新レース」で選ぶと、horse_history に馬がいない場合に
#    0頭になって意味不明な失敗をする。**実際に馬がいるレース**を選ぶこと。
c = sqlite3.connect(f'{BASE_DIR}/data/history.db')
c.row_factory = sqlite3.Row
pick = c.execute("""
    SELECT r.race_id FROM race_history r
    JOIN horse_history h ON h.race_id = r.race_id
    GROUP BY r.race_id HAVING COUNT(*) >= 8
    ORDER BY r.date DESC LIMIT 1
""").fetchone()
assert pick is not None, '8頭以上そろったレースが history.db にありません（セル3を確認）'
rid = pick[0]
rows = c.execute('SELECT * FROM horse_history WHERE race_id=? ORDER BY horse_num', (rid,)).fetchall()
rr = c.execute('SELECT * FROM race_history WHERE race_id=?', (rid,)).fetchone()
c.close()
assert rows, f'{rid} の出走馬が取得できません'

from src.scraper.jra_scraper import get_history_from_db
hist_path = f'{BASE_DIR}/data/history.db'
horses = []
for r in rows:
    horses.append({
        'name': r['horse_name'], 'horse_num': r['horse_num'],
        'post_position': r['horse_num'], 'jockey': r['jockey'] or '',
        'trainer': r['trainer'] or '', 'sex': r['sex'] or '牡',
        'age': r['age'] or 4, 'weight_load': r['weight_load'] or 56.0,
        'win_odds': None, 'popularity': r['popularity'] or 0,
        'history': get_history_from_db(r['horse_name'], hist_path),
    })

# calc_all は race 辞書ひとつを受け取り、出走馬は race['horses'] に入れる
race = {'race_id': rid, 'id': rid, 'date': rr['date'],
        'racecourse': rr['racecourse'], 'distance': rr['distance'],
        'surface': rr['surface'], 'track_condition': rr['track_condition'] or '良',
        'race_class': rr['race_class'] or '1勝クラス', 'race_num': rr['race_num'],
        'horses': horses}

scored = calc_all(race)
tot = sum(h.get('win_prob', 0) for h in scored)
print(f'\n  レース {rid}: {len(scored)}頭')
print(f'  win_prob 合計 = {tot:.4f}  (1.0 付近であること)')
assert len(scored) == len(horses), f'calc_all が {len(scored)}頭しか返していません'
assert 0.95 < tot < 1.05, 'win_prob が正規化されていません'
assert any(h.get('ability_margin') is not None for h in scored), \
    'ability_margin が None＝残差モデル経路を通っていません'
for h in sorted(scored, key=lambda x: -x.get('win_prob', 0))[:3]:
    print(f'    {h.get("name","?"):16s} win={h.get("win_prob",0)*100:5.1f}% '
          f'cal={h.get("cal_prob",0):.3f} ability={h.get("ability_margin")}')
print('\n✅ 統合テスト通過')


In [ ]:
# == セル9: GitHub main にプッシュ =========================================
# 失敗したとき「何が原因か」がその場で分かるようにしてある。
import base64, json as _json, os as _os

import requests

REPO = 'hanagenuku/keiba_ai'
FILES = [
    'data/xgb_fukusho_model.pkl', 'data/xgb_feature_cols.json',
    'data/xgb_calibrator.pkl', 'data/pace_model.pkl',
    'data/jockey_pace_stats.json', 'data/speed_index_cache.pkl',
    'data/xgb_fukusho_model_residual.pkl', 'data/xgb_feature_cols_residual.json',
    'data/retrain_v6_report.json',
]

# ── 0. PAT の取得（ここで落ちるケースが多いので個別に扱う）──────────────
GITHUB_PAT = None
try:
    from google.colab import userdata
    GITHUB_PAT = userdata.get('GITHUB_PAT')
except Exception as e:
    print(f'❌ GITHUB_PAT を取得できません: {type(e).__name__}: {e}')
    print('   Colab左の🔑（シークレット）で GITHUB_PAT を登録し、')
    print('   このノートブックからのアクセスを有効化してください。')
if not GITHUB_PAT:
    raise SystemExit('GITHUB_PAT が未設定のため中止しました（モデルは Drive に保存済みです）')

# ── 1. プリフライト: 本番ファイルが「今回学習したモデル」か確認 ──────────
# セル7を飛ばして9を実行すると、旧モデルをpushしてしまう。
prod = f'{BASE_DIR}/data/xgb_feature_cols.json'
res  = f'{BASE_DIR}/data/xgb_feature_cols_residual.json'
if not _os.path.exists(prod) or not _os.path.exists(res):
    raise SystemExit('本番/残差モデルのメタが見つかりません。セル5〜7を先に実行してください。')
with open(prod) as f:
    meta = _json.load(f)
with open(res) as f:
    meta_res = _json.load(f)
if meta.get('trained_at') != meta_res.get('trained_at'):
    print('⚠ 本番ファイルが今回の学習結果と一致しません。')
    print(f'   本番: trained_at={meta.get("trained_at")} AUC={meta.get("val_auc")}')
    print(f'   残差: trained_at={meta_res.get("trained_at")} AUC={meta_res.get("val_auc")}')
    raise SystemExit('セル7（本番切替）を実行してから、もう一度このセルを実行してください。')

bad = [c for c in meta.get('feature_cols', []) if 'member_level' in c or 'pred_gap' in c]
assert not bad, f'リーク特徴量がモデルに残っています: {bad[:5]}'

auc = meta.get('val_auc')
msg = (f'model: retrain v6 {len(meta["feature_cols"])}feat '
       f'AUC={auc:.4f} (residual+drift)' if isinstance(auc, (int, float))
       else f'model: retrain v6 {len(meta["feature_cols"])}feat (residual+drift)')
print(f'コミットメッセージ: {msg}\n')

# ── 2. push（失敗時は HTTP ステータスと本文を必ず出す）───────────────────
def push_file(relpath, pat, message):
    path = f'{BASE_DIR}/{relpath}'
    if not _os.path.exists(path):
        print(f'SKIP  {relpath} (ファイルなし)')
        return False
    url = f'https://api.github.com/repos/{REPO}/contents/{relpath}'
    h = {'Authorization': f'token {pat}',
         'Accept': 'application/vnd.github.v3+json'}
    try:
        r = requests.get(url, headers=h, params={'ref': 'main'}, timeout=60)
        sha = r.json().get('sha') if r.status_code == 200 else None
        with open(path, 'rb') as f:
            content = base64.b64encode(f.read()).decode()
        payload = {'message': message, 'content': content, 'branch': 'main'}
        if sha:
            payload['sha'] = sha
        r = requests.put(url, headers=h, json=payload, timeout=180)
    except Exception as e:
        print(f'NG    {relpath} 通信エラー: {type(e).__name__}: {e}')
        return False
    if r.status_code in (200, 201):
        print(f'OK    {relpath}  ({_os.path.getsize(path):,} bytes)')
        return True
    # 失敗理由をそのまま出す（401=PAT無効/期限切れ, 403=権限不足, 409=競合）
    try:
        detail = r.json().get('message', r.text[:200])
    except Exception:
        detail = r.text[:200]
    print(f'NG({r.status_code}) {relpath}')
    print(f'      -> {detail}')
    if r.status_code == 401:
        print('      -> PATが無効か期限切れです。再発行してシークレットを更新してください。')
    elif r.status_code == 403:
        print('      -> PATに contents:write 権限があるか確認してください。')
    elif r.status_code == 409:
        print('      -> 競合。週次ワークフロー実行中の可能性。数分後に再実行してください。')
    return False

results = {rel: push_file(rel, GITHUB_PAT, msg) for rel in FILES}
pushed = sum(results.values())
print(f'\n完了: {pushed}/{len(FILES)} ファイルをpush')

failed = [k for k, v in results.items() if not v and _os.path.exists(f'{BASE_DIR}/{k}')]
if failed:
    print(f'⚠ 失敗: {failed}')
    print('  上のエラー本文を確認してください。')
else:
    print('\n>>> 次回の週末ワークフローから新モデルで予想が生成されます。')
    print('    数週間後に data/kpi_weekly.json の delta を確認してください。')
